## Loading data

In [26]:
# Importing the necessary libraries
import pandas as pd
import numpy as np
import kennard_stone as ks
pd.options.plotting.backend = 'plotly'  # setting plotly as the backend for pandas plotting

# Add parent directory to sys.path so local module 'synthetic' (one level up) can be imported
import sys
from pathlib import Path # for path manipulations
parent_dir = Path.cwd().parent.parent.resolve() # move two levels up from current working directory
if str(parent_dir) not in sys.path: # check to avoid duplicates
    sys.path.insert(0, str(parent_dir)) # insert at the start of sys.path to prioritize local modules

# Loading a soil spectral dataset based on X-ray fluorescence (XRF)
data_complete = pd.read_csv(f'{parent_dir}/XRF_databases/bank_notes/plsda/bank_notes.csv', sep=';') # local copy of Toledo 2022 dataset (os ... indica para omitir o caminho longo)
data = data_complete.loc[:, '1':'26.07']

In [27]:
# Split dataset by class and create calibration/prediction sets using Kennard-Stone (as in original pipeline)
data_A = data_complete[data_complete['Class'] == 'A'].reset_index(drop=True)
data_B = data_complete[data_complete['Class'] == 'B'].reset_index(drop=True)

# splitting the data into calibration and prediction sets by kennard-stone algorithm
XA_cal, XA_pred = ks.train_test_split(data_A.loc[:, '1':'26.07'], test_size=0.30)  # class A
XA_cal = XA_cal.reset_index(drop=True)
XA_pred = XA_pred.reset_index(drop=True)

XB_cal, XB_pred = ks.train_test_split(data_B.loc[:, '1':'26.07'], test_size=0.30)  # class B
XB_cal = XB_cal.reset_index(drop=True)
XB_pred = XB_pred.reset_index(drop=True)

Xcalclass = pd.concat([XA_cal, XB_cal], axis=0).reset_index(drop=True)  # concatenating both classes
Xpredclass = pd.concat([XA_pred, XB_pred], axis=0).reset_index(drop=True)
ycalclass = pd.Series(['A']*XA_cal.shape[0] + ['B']*XB_cal.shape[0])  # target for calibration set
ypredclass = pd.Series(['A']*XA_pred.shape[0] + ['B']*XB_pred.shape[0])  # target for prediction set

# preprocessings
import preprocessings as prepr  # preprocessing methods for XRF data

Xcalclass_prep, mean_calclass, mean_calclass_poisson  = prepr.poisson(Xcalclass, mc=True)
Xpredclass_prep = ((Xpredclass/np.sqrt(mean_calclass)) - mean_calclass_poisson)

2026-01-22 16:38:20,945 - kennard_stone.utils._pairwise:114[INFO] - Calculating pairwise distances using scikit-learn.

2026-01-22 16:38:20,949 - kennard_stone.utils._pairwise:114[INFO] - Calculating pairwise distances using scikit-learn.



2026-01-22 16:38:21,045 - kennard_stone.utils._pairwise:114[INFO] - Calculating pairwise distances using scikit-learn.

2026-01-22 16:38:21,048 - kennard_stone.utils._pairwise:114[INFO] - Calculating pairwise distances using scikit-learn.



In [28]:
# PLS-DA with optimized latent variables
from modeling import pls_optimized

plsda_results = pls_optimized(
    Xcalclass_prep, 
    ycalclass,
    LVmax=4,
    Xpred=Xpredclass_prep,
    ypred=ypredclass,
    aim='classification',
    cv=10
)

# Convenience references used later
pls_model = plsda_results[3]               # fitted PLS model
vip_scores_mat = plsda_results[4]          # VIP scores matrix (features × LV)
y_pred_cont = plsda_results[5].iloc[:, -1] # continuous predictions for Xcalclass (used for MI/Cov)

# plotando o vip scores rapidamente
vip_scores_mat.T.plot()

In [31]:
# establishing spectral cuts based on expert knowledge of XRF spectra
spectral_cuts = [
('background1', 1.0, 2.74),
('Ar ka + Ag L', 2.76, 3.47),
('Ca ka', 3.5, 3.91),
('Ca kb', 3.93, 4.24),
('Ti ka', 4.26, 4.72),
('Ti kb', 4.75, 5.13),
('background2', 5.16, 6.12),
('Fe ka', 6.15, 6.76),
('Fe kb', 6.79, 7.32),
('background3', 7.35, 7.78),
('Cu', 7.81, 8.29),
('background4', 8.32, 21.46),
('Ag ka scattering', 21.49, 22.71),
('background5', 22.74, 24.52),
('background6', 24.55, 26.07),
]

import explaining as exp
spectral_zones_class = exp.extract_spectral_zones(Xcalclass_prep, spectral_cuts)
zone_sums_df = exp.aggregate_spectral_zones(spectral_zones_class, aggregator='extreme')
predicates_quantiles = exp.predicates_by_quantiles(zone_sums_df, [0.2, 0.4, 0.6, 0.8])
co_occurrence_matrix_df = predicates_quantiles[2]
predicate_info_dict = exp.create_predicate_info_dict(
    predicates_df=predicates_quantiles[0],
    predicate_indicator_df=predicates_quantiles[1],
    zone_aggregated_df=zone_sums_df,
    y_predicted_numeric=y_pred_cont
)

## VIP, Regression Coefficients e SHAP (como no original)

In [32]:
# VIP scores por energia
vip_scores_df = pd.DataFrame({
    'energy': vip_scores_mat.T.index,
    'VIP_Score': vip_scores_mat.T.iloc[:,0].values
})
vip_scores_df = vip_scores_df.sort_values(by='VIP_Score', ascending=False).reset_index(drop=True)
energy_to_zone_vip = {}
for zone_name, start, end in spectral_cuts:
    for e in vip_scores_df['energy']:
        ef = float(e)
        if start <= ef <= end:
            energy_to_zone_vip[e] = zone_name
vip_scores_df['Zone'] = vip_scores_df['energy'].map(energy_to_zone_vip)
vip_scores_unique_df = vip_scores_df.drop_duplicates(subset=['Zone'], keep='first').reset_index(drop=True)
vip_scores_unique_df = vip_scores_unique_df.sort_values(by='VIP_Score', ascending=False).reset_index(drop=True)

# Coeficientes de regressão do PLS
reg_vet = pd.DataFrame(pls_model.coef_, columns=pls_model.feature_names_in_).T
reg_vet.insert(0, 'energy', reg_vet.index)
reg_vet = reg_vet.reset_index(drop=True)
reg_vet.columns = ['energy','Reg_coef']
reg_vet['Abs_Reg_coef'] = reg_vet['Reg_coef'].abs()
reg_vet = reg_vet.sort_values(by='Abs_Reg_coef', ascending=False).reset_index(drop=True)
energy_to_zone_reg = {}
for zone_name, start, end in spectral_cuts:
    for e in reg_vet['energy']:
        ef = float(e)
        if start <= ef <= end:
            energy_to_zone_reg[e] = zone_name
reg_vet['Zone'] = reg_vet['energy'].map(energy_to_zone_reg)
reg_vet_unique_df = reg_vet.drop_duplicates(subset=['Zone'], keep='first').reset_index(drop=True)
reg_vet_unique_df = reg_vet_unique_df.sort_values(by='Abs_Reg_coef', ascending=False).reset_index(drop=True)

# vamos agora extrair as variaveis mais importantes atraves do método SHAP
# import shap

# # Para PLSRegression, usamos KernelExplainer porque não há explainer dedicado muito rápido
# explainer_pls = shap.KernelExplainer(plsda_results[3].predict, Xcalclass_prep)
# shap_values_pls = explainer_pls(Xcalclass_prep)

# shap_global_importance = pd.DataFrame({
#     'energy': Xpredclass_prep.columns,
#     'Mean_Abs_SHAP': np.abs(shap_values_pls.values).mean(axis=0)}) # tomando a importancia global como a media dos valores absolutos dos valores SHAP para cada feature
# shap_global_importance.sort_values(by='Mean_Abs_SHAP', ascending=False, inplace=True)

# # vamos gerar uma nova coluna em shap_global_importance com o nome da zona espectral correspondente de acordo com a lista spectral_cuts
# energy_to_zone_shap = {}
# for zone_name, start, end in spectral_cuts:
#     for i in shap_global_importance['energy']:
#         i_float = float(i)
#         if start <= i_float <= end:
#             energy_to_zone_shap[i] = zone_name
# shap_global_importance['Zone'] = shap_global_importance['energy'].map(energy_to_zone_shap)

# # agora vamos filtrar shap_global_importance para manter apenas as zonas espectrais únicas com maior SHAP score
# shap_unique_df = shap_global_importance.drop_duplicates(subset=['Zone'], keep='first').reset_index(drop=True)
# shap_unique_df = shap_unique_df.sort_values(by='Mean_Abs_SHAP', ascending=False).reset_index(drop=True)
# shap_unique_df.to_csv('shap_bank_notes.csv', index=False, sep=';')
shap_unique_df = pd.read_csv('shap_bank_notes.csv', sep=';') # loading previously saved shap_unique_df

# **bagging - covariance**

In [33]:
import explaining as exp

# LISTA DE SEMENTES A TESTAR
random_seeds = [0, 1, 2]

all_results_cov = {}
training_samples = len(Xcalclass)

# LOOP: PROCESSAR CADA SEMENTE
y_predicted_numeric = plsda_results[5].iloc[:, -1] # predições numéricas do modelo

for seed in random_seeds:
    print(f"\n{'='*70}")
    print(f"Processando semente: {seed}")
    print(f"{'='*70}\n")
    # Bagging
    bags_result_seed = exp.bagging_predicates(
        zone_sums_df=zone_sums_df,
        y_predicted_numeric=y_predicted_numeric,
        predicates_df=predicates_quantiles[0],
        n_bags=20,
        #n_predicates_per_bag=40,
        n_samples_per_bag=int(training_samples*0.5), # 80 % da base para amostrar (convertido para int)
        min_samples_per_predicate=int(training_samples*0.2), # 20 % da base para limitar (convertido para int)
        replace=False,
        sample_bagging=True,
        predicate_bagging=False,
        random_seed=seed
    )
    # Inserir classe prevista
    for bag_name, pred_dict in bags_result_seed.items(): # iterando sobre cada bag
        for pred_rule, df_info in pred_dict.items():
            df_info['Class_Predicted'] = np.where(df_info['Predicted_Y'] >= 0.5, 'A', 'B') # binarizando com threshold 0.5, A = eut, B = dist
    # Calcular MI
    cov_results_dict_seed = exp.calculate_predicate_metrics(
        bags_result=bags_result_seed,
        metric='covariance', # covariance ou mutual_information
        threshold=0.001, # threshold para cortar predicados irrelevantes
        n_neighbors=5
    )
    # Salvar no dicionário principal
    all_results_cov[seed] = {
        'bags_result': bags_result_seed,
        'cov_results_dict': cov_results_dict_seed
    }

# CONSTRUÇÃO DE GRAFOS PARA MÚLTIPLAS SEMENTES (LOOP EXTERNO)
# Dicionário para armazenar grafos
graphs_by_seed = {}

for seed in random_seeds:
    print(f"\n{'='*70}")
    print(f"Processando Grafo - Semente: {seed}")
    print(f"{'='*70}\n")
    # Construir grafo para esta semente
    DG = exp.build_predicate_graph(
        bags_result=all_results_cov[seed]['bags_result'],
        mi_results_dict=all_results_cov[seed]['cov_results_dict'],
        co_occurrence_matrix_df=co_occurrence_matrix_df,
        predicates_df=predicates_quantiles[0],
        random_state=seed,
        show_details=True
    )
    # Armazenar grafo
    graphs_by_seed[seed] = DG  

# Calcular LRC usando a função pronta do explaining.py
lrc_cov_by_seed = {}
for seed in random_seeds:
    DG = graphs_by_seed[seed]
    lrc_cov_df_seed = exp.calculate_lrc_single_graph(DG, predicates_quantiles[0])
    lrc_cov_df_seed['Seed'] = seed  # Adicionar coluna com a semente
    lrc_cov_by_seed[seed] = lrc_cov_df_seed

# junando todas as colunas 'Node' de lrc_by_seed em um único dataframe
lrc_cov_all_seeds_df = pd.DataFrame()
for seed in random_seeds:
    lrc_cov_df_seed = lrc_cov_by_seed[seed].rename(columns={'Node': f'Predicate_Cov_Seed_{seed}'})
    lrc_cov_all_seeds_df = pd.concat([lrc_cov_all_seeds_df, lrc_cov_df_seed[[f'Predicate_Cov_Seed_{seed}']]], axis=1)

# vamos filtrar lrc_by_seed em cada semente para manter apenas as zonas espectrais únicas com maior LRC em um mesmo dataframe
lrc_cov_unique_by_seed = {}
for seed, lrc_df in lrc_cov_by_seed.items():
    lrc_cov_unique_df = lrc_df.drop_duplicates(subset=['Zone'], keep='first').reset_index(drop=True)
    lrc_cov_unique_df = lrc_cov_unique_df.sort_values(by='Local_Reaching_Centrality', ascending=False).reset_index(drop=True)
    lrc_cov_unique_by_seed[seed] = lrc_cov_unique_df

lrc_cov_all_seeds_df # exibindo o dataframe consolidado com predicados de todas as sementes


Processando semente: 0

Bag_1 | Amostras: Sim | Predicados: Não (Todos) | Válidos: 79 | Descartados: 41
Bag_2 | Amostras: Sim | Predicados: Não (Todos) | Válidos: 78 | Descartados: 42
Bag_3 | Amostras: Sim | Predicados: Não (Todos) | Válidos: 76 | Descartados: 44
Bag_4 | Amostras: Sim | Predicados: Não (Todos) | Válidos: 76 | Descartados: 44
Bag_5 | Amostras: Sim | Predicados: Não (Todos) | Válidos: 76 | Descartados: 44
Bag_6 | Amostras: Sim | Predicados: Não (Todos) | Válidos: 77 | Descartados: 43
Bag_7 | Amostras: Sim | Predicados: Não (Todos) | Válidos: 76 | Descartados: 44
Bag_8 | Amostras: Sim | Predicados: Não (Todos) | Válidos: 79 | Descartados: 41
Bag_9 | Amostras: Sim | Predicados: Não (Todos) | Válidos: 78 | Descartados: 42
Bag_10 | Amostras: Sim | Predicados: Não (Todos) | Válidos: 79 | Descartados: 41
Bag_11 | Amostras: Sim | Predicados: Não (Todos) | Válidos: 83 | Descartados: 37
Bag_12 | Amostras: Sim | Predicados: Não (Todos) | Válidos: 77 | Descartados: 43
Bag_13 | Amo

/home/jvribeiro/.local/lib/python3.10/site-packages/networkx/algorithms/centrality/reaching.py:193: RuntimeWarning:

divide by zero encountered in scalar divide

/home/jvribeiro/.local/lib/python3.10/site-packages/networkx/algorithms/centrality/reaching.py:193: RuntimeWarning:

divide by zero encountered in scalar divide




Processando LRC do grafo...

Processando LRC do grafo...


/home/jvribeiro/.local/lib/python3.10/site-packages/networkx/algorithms/centrality/reaching.py:193: RuntimeWarning:

divide by zero encountered in scalar divide



,Predicate_Cov_Seed_0,Predicate_Cov_Seed_1,Predicate_Cov_Seed_2
0,Fe ka > -9.53,Fe ka > -9.53,Fe ka > -9.53
1,Fe ka > -9.80,Fe ka > -9.80,Fe ka > -9.80
2,Ca ka > -9.12,Ca ka > -9.12,Ca kb > -3.42
3,Fe ka > -10.00,Fe ka > -10.00,Ca ka > -9.12
4,Ti ka <= 5.26,Ca kb > -3.42,Ti ka > -11.81
...,...,...,...
87,Ca kb <= -3.42,background1 > 2.53,background6 > 2.71
88,Fe kb <= -3.54,background2 > 2.59,Fe ka <= -9.80
89,background2 > 2.59,Fe kb <= -3.54,Cu <= -4.31
90,Class_A,Class_A,Class_A


# **Perturbation**

In [34]:
import explaining as exp
import permutation as perm

# LISTA DE SEMENTES A TESTAR
random_seeds = [0, 1, 2]

all_results_pert = {}
training_samples = len(Xcalclass)

# LOOP: PROCESSAR CADA SEMENTE
y_predicted_numeric = plsda_results[5].iloc[:, -1] # predições numéricas do modelo

for seed in random_seeds:
    print(f"\n{'='*70}")
    print(f"Processando semente: {seed}")
    print(f"{'='*70}\n")
    # Bagging
    bags_result_seed = exp.bagging_predicates(
        zone_sums_df=zone_sums_df,
        y_predicted_numeric=y_predicted_numeric,
        predicates_df=predicates_quantiles[0],
        n_bags=10,
        #n_predicates_per_bag=40,
        n_samples_per_bag=int(training_samples*0.8), # 80 % da base para amostrar (convertido para int)
        min_samples_per_predicate=int(training_samples*0.2), # 20 % da base para limitar (convertido para int)
        replace=False,
        sample_bagging=True,
        predicate_bagging=False,
        random_seed=seed
    )
    # Inserir classe prevista
    for bag_name, pred_dict in bags_result_seed.items(): # iterando sobre cada bag
        for pred_rule, df_info in pred_dict.items():
            df_info['Class_Predicted'] = np.where(df_info['Predicted_Y'] >= 0.5, 'A', 'B') # binarizando com threshold 0.5, A = eut, B = dist

    pert_results_seed = perm.calculate_predicate_perturbation(
        estimator=pls_model,
        Xcalclass_prep=Xcalclass_prep,
        folds_struct=bags_result_seed,
        predicates_df=predicates_quantiles[0],
        spectral_cuts=spectral_cuts,
        perturbation_value=np.mean(y_predicted_numeric)*0,
        metric='mean_relative_dev',   # Média com sinal (pode ser negativo)
        verbose=True
    )

    # Remove todos os valores iguais a zero de todos os bags em perm_results[bag]["Permutation"] e salva como perm_results_thresholded
    # pert_results_seed_thresholded = {}
    # for bag, df in perm_results_seed.items():
    #     # Verifica se é um DataFrame e se a coluna 'Permutation' existe
    #     if isinstance(df, pd.DataFrame) and 'Permutation' in df.columns:
    #         filtered_df = df[df['Permutation'] > 0].copy()
    #         pert_results_seed_thresholded[bag] = filtered_df
    #     else:
    #         # Se não for DataFrame esperado, apenas copia
    #         pert_results_seed_thresholded[bag] = df

    # Salvar no dicionário principal
    all_results_pert[seed] = {
        'bags_result': bags_result_seed,
        'pert_results_dict': pert_results_seed
    }

# CONSTRUÇÃO DE GRAFOS PARA MÚLTIPLAS SEMENTES (LOOP EXTERNO)
# Dicionário para armazenar grafos
graphs_pert_by_seed = {}

for seed in random_seeds:
    print(f"\n{'='*70}")
    print(f"Processando Grafo - Semente: {seed}")
    print(f"{'='*70}\n")
    # Construir grafo para esta semente
    DG = exp.build_predicate_graph(
        bags_result=all_results_pert[seed]['bags_result'],
        mi_results_dict=all_results_pert[seed]['pert_results_dict'],
        co_occurrence_matrix_df=co_occurrence_matrix_df,
        predicates_df=predicates_quantiles[0],
        random_state=seed,
        show_details=True
    )
    # Armazenar grafo
    graphs_pert_by_seed[seed] = DG  

# Calcular LRC usando a função pronta do explaining.py
lrc_pert_by_seed = {}
for seed in random_seeds:
    DG = graphs_pert_by_seed[seed]
    lrc_pert_df_seed = exp.calculate_lrc_single_graph(DG, predicates_quantiles[0])
    lrc_pert_df_seed['Seed'] = seed  # Adicionar coluna com a semente
    lrc_pert_by_seed[seed] = lrc_pert_df_seed

# junando todas as colunas 'Node' de lrc_by_seed em um único dataframe
lrc_pert_all_seeds_df = pd.DataFrame()
for seed in random_seeds:
    lrc_pert_df_seed = lrc_pert_by_seed[seed].rename(columns={'Node': f'Predicate_pert_Seed_{seed}'})
    lrc_pert_all_seeds_df = pd.concat([lrc_pert_all_seeds_df, lrc_pert_df_seed[[f'Predicate_pert_Seed_{seed}']]], axis=1)

# vamos filtrar lrc_by_seed em cada semente para manter apenas as zonas espectrais únicas com maior LRC em um mesmo dataframe
lrc_pert_unique_by_seed = {}
for seed, lrc_df in lrc_pert_by_seed.items():
    lrc_pert_unique_df = lrc_df.drop_duplicates(subset=['Zone'], keep='first').reset_index(drop=True)
    lrc_pert_unique_df = lrc_pert_unique_df.sort_values(by='Local_Reaching_Centrality', ascending=False).reset_index(drop=True)
    lrc_pert_unique_by_seed[seed] = lrc_pert_unique_df

lrc_pert_all_seeds_df # exibindo o dataframe consolidado com predicados de todas as sementes


Processando semente: 0

Bag_1 | Amostras: Sim | Predicados: Não (Todos) | Válidos: 91 | Descartados: 29
Bag_2 | Amostras: Sim | Predicados: Não (Todos) | Válidos: 91 | Descartados: 29
Bag_3 | Amostras: Sim | Predicados: Não (Todos) | Válidos: 90 | Descartados: 30
Bag_4 | Amostras: Sim | Predicados: Não (Todos) | Válidos: 90 | Descartados: 30
Bag_5 | Amostras: Sim | Predicados: Não (Todos) | Válidos: 90 | Descartados: 30
Bag_6 | Amostras: Sim | Predicados: Não (Todos) | Válidos: 91 | Descartados: 29
Bag_7 | Amostras: Sim | Predicados: Não (Todos) | Válidos: 90 | Descartados: 30
Bag_8 | Amostras: Sim | Predicados: Não (Todos) | Válidos: 90 | Descartados: 30
Bag_9 | Amostras: Sim | Predicados: Não (Todos) | Válidos: 90 | Descartados: 30
Bag_10 | Amostras: Sim | Predicados: Não (Todos) | Válidos: 90 | Descartados: 30
PERTURBATION IMPORTANCE PARA PREDICADOS
Valor de perturbação: 0.0
Métrica: mean_relative_dev
Total de folds: 10


[Bag_1] Processando 91 predicados...
  Predicado: background

/home/jvribeiro/.local/lib/python3.10/site-packages/networkx/algorithms/centrality/reaching.py:193: RuntimeWarning:

divide by zero encountered in scalar divide

/home/jvribeiro/.local/lib/python3.10/site-packages/networkx/algorithms/centrality/reaching.py:193: RuntimeWarning:

divide by zero encountered in scalar divide




Processando LRC do grafo...


/home/jvribeiro/.local/lib/python3.10/site-packages/networkx/algorithms/centrality/reaching.py:193: RuntimeWarning:

divide by zero encountered in scalar divide



,Predicate_pert_Seed_0,Predicate_pert_Seed_1,Predicate_pert_Seed_2
0,Ti ka > 5.26,Ti ka > 5.26,Ti ka > -3.08
1,Ti ka > -3.08,Ti ka > -3.08,Ti ka > 5.26
2,background4 <= 3.92,background1 <= 2.22,Ti ka > -11.81
3,background2 <= 3.28,Ti ka <= -3.08,Ca ka <= -3.71
4,background1 <= 2.22,Cu > -4.31,Ti ka <= 9.71
...,...,...,...
89,background2 <= 2.16,Ca kb <= -3.42,Ag ka scattering <= -1.94
90,Fe ka <= -10.00,background5 > 2.48,background4 <= 3.17
91,Class_A,Ag ka scattering > 2.57,Class_A
92,Class_B,Class_A,Class_B


In [35]:
all_results_pert[0]['pert_results_dict']['Bag_1']

,Predicate,Perturbation
0,Ti ka > 5.26,1.526886
1,Ti ka > -3.08,0.780562
2,Ti ka > -11.81,0.660514
3,Ti ka <= 9.71,0.364703
4,Ti ka <= -3.08,0.333393
...,...,...
86,background6 > -2.43,0.000332
87,background3 <= 2.47,0.000313
88,Ar ka + Ag L <= 2.08,0.000278
89,background3 > -1.63,0.000209


In [36]:
from collections import defaultdict

# 1. Coletar posições de cada predicado em cada bag
positions_dict = defaultdict(list)

for bag_num in range(1, 11):
    bag_name = f'Bag_{bag_num}'
    bag_df = all_results_pert[0]['pert_results_dict'][bag_name]
    
    for position, predicate in enumerate(bag_df['Predicate'], start=1):
        positions_dict[predicate].append(position)

# 2. Calcular média e número de aparições
results = []
for predicate, positions in positions_dict.items():
    results.append({
        'Predicate': predicate,
        'Mean_Position': np.mean(positions),
        'Appearances': len(positions),
        'Zone' : predicates_quantiles[0].loc[predicates_quantiles[0]['rule'] == predicate, 'zone'].values[0]
    })

# 3. Ordenar: menor posição média primeiro, mais aparições em caso de empate
ranking_df = pd.DataFrame(results).sort_values(
    by=['Mean_Position', 'Appearances'], 
    ascending=[True, False]
).reset_index(drop=True)

# 4. Lista final ordenada
lista_ordenada = ranking_df['Predicate'].tolist()
ranking_df

,Predicate,Mean_Position,Appearances,Zone
0,Ti ka > 5.26,1.3,10,Ti ka
1,Ti ka > -3.08,3.4,10,Ti ka
2,Ti ka > -11.81,3.6,10,Ti ka
3,Ti ka <= -3.08,3.8,10,Ti ka
4,Cu > -3.81,6.9,10,Cu
...,...,...,...,...
86,Ar ka + Ag L <= 2.08,82.5,10,Ar ka + Ag L
87,background3 <= 2.09,83.2,10,background3
88,background3 <= 1.70,83.8,10,background3
89,background3 <= 2.47,85.0,10,background3


In [37]:
ranking_unique_df = ranking_df.drop_duplicates(subset=['Zone'], keep='first').reset_index(drop=True)
ranking_unique_df['Zone']

0                Ti ka
1                   Cu
2                Ca ka
3                Ti kb
4     Ag ka scattering
5                Ca kb
6                Fe ka
7          background4
8          background6
9          background1
10         background2
11               Fe kb
12        Ar ka + Ag L
13         background5
14         background3
Name: Zone, dtype: object

In [40]:
import numpy as np

max_len = max(
    len(vip_scores_unique_df['Zone']),
    len(reg_vet_unique_df['Zone']),
    len(shap_unique_df['Zone']),
    #len(lrc_cov_unique_df['Zone']),
    len(lrc_pert_unique_df['Zone'])
)

def pad_list(lst, length):
    return list(lst) + [None] * (length - len(lst))

features_importance = pd.DataFrame({
    'Vip': pad_list(vip_scores_unique_df['Zone'], max_len),
    'Reg_coef': pad_list(reg_vet_unique_df['Zone'], max_len),
    'Shap': pad_list(shap_unique_df['Zone'].astype(str), max_len),
    'Ranking' : pad_list(ranking_unique_df['Zone'], max_len)
})

for seed, lrc_unique_df in lrc_cov_unique_by_seed.items():
    features_importance[f'LRC_cov_{seed}'] = pad_list(lrc_unique_df['Zone'].iloc[:10].tolist(), max_len)

# for seed, lrc_unique_df in lrc_pert_unique_by_seed.items():
#     features_importance[f'LRC_pert_{seed}'] = pad_list(lrc_unique_df['Zone'].iloc[:10].tolist(), max_len)

# vamos exportar o df features_importance para um arquivo excel onde vamos nomear a sheet de acordo com as comparacoes feitas
#features_importance.to_excel('features_importance_soil_vnir.xlsx', index=False, sheet_name='Perm')
features_importance.head(20)

,Vip,Reg_coef,Shap,Ranking,LRC_cov_0,LRC_cov_1,LRC_cov_2
0,Fe ka,Ti ka,Ti ka,Ti ka,Fe ka,Fe ka,Fe ka
1,Ca ka,Ti kb,Ca ka,Cu,Ca ka,Ca ka,Ca kb
2,Ti ka,Ag ka scattering,Cu,Ca ka,Ti ka,Ca kb,Ca ka
3,Cu,Ca ka,Fe ka,Ti kb,Ca kb,Ti ka,Ti ka
4,Fe kb,Cu,Ti kb,Ag ka scattering,Fe kb,Fe kb,Ti kb
5,Ca kb,background6,Ag ka scattering,Ca kb,Ti kb,Cu,Cu
6,Ti kb,Ca kb,background6,Fe ka,Cu,Ti kb,Fe kb
7,Ag ka scattering,background4,background4,background4,background4,background4,background2
8,background6,background2,background5,background6,background6,background5,background3
9,background4,Ar ka + Ag L,background1,background1,background1,background1,background5


In [41]:
# RBO (Rank-Biased Overlap) para comparar rankings
import rbo
rbo_results = {}
reference_list = [x for x in features_importance['Vip'].tolist() if x is not None]
methods = ['Reg_coef', 'Shap', 'Ranking'] + [f'LRC_cov_{seed}' for seed in random_seeds] #+ [f'LRC_perm_{seed}' for seed in random_seeds]
for method in methods:
    compare_list = [x for x in features_importance[method].tolist() if x is not None]
    # Truncate both lists to the same length (minimum of both)
    min_len = min(len(reference_list), len(compare_list))
    ref_trunc = reference_list[:min_len]
    cmp_trunc = compare_list[:min_len]
    score = rbo.RankingSimilarity(ref_trunc, cmp_trunc).rbo(p=0.7, k=10)
    rbo_results[method] = score
rbo_results = pd.DataFrame(list(rbo_results.items()), columns=['Method','RBO_Score'])
rbo_results.insert(0, 'Reference', 'Vip')
rbo_results.sort_values(by='RBO_Score', ascending=False, inplace=True)
#rbo_results.to_excel('rbo_soil_vnir.xlsx', index=False, sheet_name='perm')
rbo_results

,Reference,Method,RBO_Score
3,Vip,LRC_cov_0,0.916997
4,Vip,LRC_cov_1,0.873269
5,Vip,LRC_cov_2,0.744249
1,Vip,Shap,0.464014
2,Vip,Ranking,0.330146
0,Vip,Reg_coef,0.235755
